# Do penny-stock explosions have a tradable conditional edge?

### A veto-filter study of exchange-listed US microcaps, 2016-2026

---

## The question, stated precisely

The published evidence against penny-stock day trading is strong and consistent.
Eraker & Ready (2015) find OTC returns are negative on average and violently
right-skewed; the SEC's own outcomes study puts the median retail OTC holding
return at **-11.9%**; Barber, Lee, Liu & Odean find **fewer than 1%** of day
traders earn predictably positive abnormal returns net of fees.

All of that measures **unconditional** expectation. That is the gap this study
examines. A filter with high precision on a *toxic sub-population* can move
$E[r \mid \text{survived}]$ even when $E[r]$ is deeply negative -- and several of
those same papers effectively hand over the filter conditions.

So the hypothesis under test is:

> **H1.** There exists a veto set $V$, computable from free data at 09:35 ET with
> no look-ahead, such that
> $$E[r_{\text{net}} \mid \neg V] > 0$$
> with a confidence interval excluding zero after multiple-testing correction, at
> a retained population fraction and capacity worth trading.

**H0** is that no such set exists. The honest prior favours H0, and this notebook
is built so rejecting H0 requires surviving every check below -- while *failing*
to reject it is a cheap, publishable result rather than a wasted month.


## Why a veto and not a predictor

The asymmetry is the whole idea.

Predicting *which* microcap will explode is close to hopeless: the outcome is
dominated by a handful of enormous winners in a fat right tail, and by
information (promotion schedules, offering timing) that is not public before the
fact. Recognising which candidates are **structurally poisoned** is far easier,
and the literature supplies most of the conditions.

Formally, partition the candidate population $C$ into a vetoed part $V$ and its
complement. Then

$$E[r \mid C] = \pi\, E[r \mid V] + (1-\pi)\, E[r \mid \neg V]$$

where $\pi = P(V)$. Rearranged:

$$E[r \mid \neg V] = \frac{E[r \mid C] - \pi\, E[r \mid V]}{1 - \pi}$$

**Lemma 1 (a veto need not predict winners).** If $E[r \mid V] < E[r \mid C]$, then
$E[r \mid \neg V] > E[r \mid C]$ for any $\pi \in (0,1)$.

*Proof.* Immediate from the identity above: subtracting a below-average mass and
renormalising raises the mean. $\square$

The practical content: a veto only has to identify a sub-population that is
*worse than average*. It never has to identify winners. That is a far weaker
requirement, and it is why this study is worth running despite the literature.

**Lemma 2 (improvement is not profitability).** Lemma 1 places no lower bound on
$E[r \mid \neg V]$. If $E[r \mid C] \ll 0$, a veto can raise the conditional mean
substantially and still leave it negative.

This is not pedantry -- for penny stocks the baseline *is* strongly negative, so
the two questions come apart in exactly the regime we are in. The notebook
therefore reports **both**: a test against zero (profitable?) and a paired test
against the unfiltered baseline (does the veto help?).


## Design of the experiment

### Population (the denominator)

A result is meaningless without saying what it is a statement *about*. The
candidate screen defines the population, and every condition is knowable at
09:30 ET:

| Condition | Value | Knowable at 09:30? |
|---|---|---|
| Opening gap | $\geq +10\%$ vs prior close | yes -- uses the opening print |
| Price | \$0.50-\$10.00, **previous unadjusted** close | yes |
| Liquidity | trailing 20-day dollar volume $\geq$ \$100k | yes -- shifted |
| History | $\geq 30$ prior bars | yes |

### Timeline

```
09:30 - 09:34   observation window   features may use ONLY these bars
09:35           decision + fill      entry pays spread + impact
09:36 - 15:55   barrier scan         first barrier touched wins
15:55           vertical barrier     forced exit
```

### Labels: triple-barrier on a minute path

Following Lopez de Prado (2018, ch. 3), adapted from daily series to minute
bars. Five assumptions, each **pessimistic by construction**:

1. **Ambiguous minutes resolve against the trade.** If a bar's high crosses the
   target *and* its low crosses the stop, the bar cannot say which came first --
   we assume the stop. Microcap minute ranges are enormous, so the optimistic
   choice would manufacture edge out of bar granularity alone.
2. **No print means no fill.** Measured directly: one \$3 name printed 52 of 391
   minutes.
3. **Stops slip, targets do not.** A triggered stop is a market order crossing a
   falling book; a target is a resting limit.
4. **Entry pays the spread** plus square-root impact.
5. **Data gaps are suspicion, not absence** -- flagged `halt_suspect`, because
   free halt history does not exist and a halted position cannot be exited.

One caveat stated plainly: the scan starts at 09:36, so a barrier touched inside
the entry bar is missed. This is the only place the pessimism is not
one-directional.


In [ ]:
import pickle, sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.io as pio

warnings.filterwarnings('ignore')
ROOT = Path.cwd()
while not (ROOT / 'research').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from research.penny import viz
from research.penny.eval import stats as pstats

pio.renderers.default = 'notebook'
pd.set_option('display.float_format', lambda v: f'{v:,.4f}')

with open(ROOT / 'data/research_penny/analysis.pkl', 'rb') as fh:
    A = pickle.load(fh)

print(f"trades      : {A['n_trades']:,}")
print(f"period      : {A['date_min']} .. {A['date_max']}")
print(f"holdout     : final {A['holdout_frac']:.0%} of sessions, sealed")


## Validity checks come first

A leakage bug does not announce itself -- it produces a **better** number, so
nothing looks wrong. "The result improved" is therefore the least reliable signal
available, and these tests run before any performance figure is read.

Each deliberately destroys the thing being measured and asserts the edge dies
with it. An edge that survives a destroyed label is an edge in the *pipeline*.

| Test | What it breaks | Expected |
|---|---|---|
| `constant_return` | all returns identical | no veto can create dispersion |
| `shuffled_labels` | feature-outcome association | FDR survivors $\approx q$, not $\gg q$ |
| `date_shift` | feature-outcome pairing, keeping day clustering | same |

> **Note on `shuffled_labels`.** This test initially *failed*, and the reason is
> instructive. The search tests each subset against zero -- correct for the real
> study. Under a shuffle every subset inherits the panel's overall mean, which for
> penny stocks is emphatically non-zero, so every subset read as "significant".
> Returns are now demeaned before the null search, which poses the question the
> null actually cares about: can the vetoes still *discriminate*?


In [ ]:
nulls = A.get('nulls')
if nulls is not None:
    display(nulls[['name', 'status', 'observed', 'reference', 'detail']])
    if nulls['passed'].all():
        print('\nAll null tests PASS -- results below are interpretable.')
    else:
        print('\nNULL TESTS FAILED -- do not interpret anything below.')


## Cross-validation design

Random K-fold is invalid here for three separate reasons, any one fatal:

1. **Temporal order.** A fold using 2026 to predict 2022 is not a forecast.
2. **Same-day clustering.** Trades in one session share a market factor. Splitting
   a day across train and test lets the model learn that day's regime from one
   half and "predict" the other. Folds are split on **whole days**.
3. **Serial correlation.** Adjacent days are not independent, hence an **embargo**.

**A distinction worth making explicitly.** Labels here are same-session, so label
horizons do **not** overlap and the classic purge for overlapping labels is
unnecessary. The embargo addresses serial correlation, not "feature-window
bleed" -- in a strictly sequential design a test day's trailing 60-day window
reaching into the train *period* is not leakage, it is exactly what live trading
does. No embargo of any length changes that. (An earlier version of this
docstring claimed otherwise and was wrong; it matters because converting to
Lopez de Prado-style purged K-fold, where train may *follow* test, would make the
requirement genuinely different.)


In [ ]:
for line in A['folds']:
    print(line)


## The candidate vetoes

Every veto is registered with the claim it encodes and its source. A veto with
no prior is a fishing expedition dressed as a hypothesis -- and it would still
inflate the multiple-testing correction for everything else.

One veto is marked **diagnostic**: `veto_delisted_soon` uses future information
by construction. It is never included in a tradable search (the search function
raises if you try); it exists only to bound how much of any edge is simply the
avoidance of terminal names -- which a live system would need filings to detect.


In [ ]:
display(A['veto_registry'])


## Baseline: what the unfiltered population earns

This is the denominator every later claim is measured against.


In [ ]:
b = A['baseline']
print(f'trades           {b.n_trades:,} over {b.n_days:,} sessions')
print(f'E[net return]    {b.mean_net:+.4%}   95% CI [{b.ci_lo:+.4%}, {b.ci_hi:+.4%}]')
print(f'median           {b.median_net:+.4%}')
print(f'hit rate         {b.hit_rate:.1%}')
print(f'P(target)        {b.p_target:.1%}    P(stop) {b.p_stop:.1%}')
print(f'Sharpe           {b.sharpe:.2f}')
print(f'max drawdown     {b.max_dd:.1%}')


### Where price actually travels

Before asking whether a filter helps, check the barriers are reachable at all.
A profit target beyond the bulk of the maximum-favourable-excursion mass is
unreachable **by construction**, no matter how good the signal.


In [ ]:
# Barrier-free excursions. An earlier version of this cell used
# max_favourable, which is measured only until a barrier fires and is
# therefore capped near the target -- it made the figure say 'price never
# travels far' when the truth is the opposite.
bf = pd.read_parquet(ROOT / 'data/research_penny/barrier_free.parquet')
bar = A['params']['barriers']
viz.excursion_density(bf, bar['profit_target'], bar['stop_loss']).show()

print('True barrier-free intraday potential:')
for x in (0.05, 0.10, 0.20, 0.50, 1.00):
    print(f'  P(MFE >= {x:>5.0%}) = {(bf["mfe_full"] >= x).mean():.4f}')
print(f'\nhold-to-close mean (gross): {bf["ret_c2c"].mean():+.4f}')


## Effect of each veto in isolation

Diagnostic only. Reading the best row off this table and calling it a result is
precisely the error the fold machinery exists to prevent -- these are in-sample
marginals over the tuning period.


In [ ]:
display(A['single'].style.format({
    'retained_frac': '{:.1%}', 'mean_kept': '{:+.3%}', 'ci_lo_kept': '{:+.3%}',
    'ci_hi_kept': '{:+.3%}', 'mean_removed': '{:+.3%}', 'lift': '{:+.3%}',
    'p_kept': '{:.3f}', 'sharpe_kept': '{:.2f}',
}))
viz.veto_lift(A['single']).show()


## The conditional expectation surface

The headline figure: the precision/recall trade-off expressed in **return space**.
Left means filtering harder; up means earning more per trade. H1 requires a point
above zero whose interval excludes zero, at an x-value worth trading.

**This is an in-sample search and is reported as such**, together with the number
of hypotheses examined. Under a pure null, the best of thousands of subsets looks
excellent -- which is why the out-of-sample section below, not this one, carries
the actual claim.


In [ ]:
table = A['search']
print(f"veto sets searched : {int(table['n_hypotheses'].iloc[0]):,}")
print(f"passing BH-FDR     : {int(table['q_pass'].sum())}")
viz.conditional_surface(table, A['baseline'].mean_net).show()


In [ ]:
cols = ['name', 'n_trades', 'retained_frac', 'mean_net', 'ci_lo', 'ci_hi',
        'p_value', 'q_pass', 'sharpe', 'hit_rate']
display(table[cols].head(15).style.format({
    'retained_frac': '{:.1%}', 'mean_net': '{:+.3%}', 'ci_lo': '{:+.3%}',
    'ci_hi': '{:+.3%}', 'p_value': '{:.3f}', 'sharpe': '{:.2f}', 'hit_rate': '{:.1%}',
}))


### The fair baseline

A veto retaining $X\%$ must beat **random** filtering that retains $X\%$.
Otherwise the apparent improvement is just a smaller, noisier sample.

The placebo drops whole *days*, matching how real vetoes behave -- many veto
conditions are day-level facts, so a trade-level placebo would be an easier bar
to clear and would flatter the result.


In [ ]:
p = A.get('placebo', {})
best = A.get('best_in_sample')
if p and best is not None:
    print(f"best in-sample set   {best['name']}")
    print(f"  mean               {best['mean_net']:+.4%}")
    print(f"  placebo mean       {p['placebo_mean']:+.4%}")
    print(f"  placebo 5-95%      [{p['placebo_p05']:+.4%}, {p['placebo_p95']:+.4%}]")
    beats = best['mean_net'] > p['placebo_p95']
    print(f"  beats placebo p95  {beats}")

d = A.get('best_vs_baseline', {})
if d:
    print(f"\npaired vs baseline   {d['delta']:+.4%} "
          f"95% CI [{d['lo']:+.4%}, {d['hi']:+.4%}]  helps={d['helps']}")


## Out-of-sample: the only quotable number

Veto sets are selected on each fold's **training days** and scored on its
**held-out days**. Everything above is exploratory.

The gap between the two bars in the figure below is the most diagnostic quantity
in the study. A large, consistent gap is the signature of selection
overfitting -- the set was chosen because it looked good in training and did not
survive contact with held-out data.


In [ ]:
wf = A['wf_summary']
display(wf[['fold', 'test_start', 'test_end', 'chosen_vetoes', 'train_mean',
            'test_mean', 'test_n', 'test_retained', 'degradation']].style.format({
    'train_mean': '{:+.3%}', 'test_mean': '{:+.3%}',
    'test_retained': '{:.1%}', 'degradation': '{:+.3%}',
}))
viz.walk_forward_stability(wf).show()


In [ ]:
o = A.get('oos')
if o is not None:
    print(f'OUT-OF-SAMPLE (stitched across folds)')
    print(f'  trades         {o.n_trades:,} over {o.n_days:,} sessions')
    print(f'  E[net]         {o.mean_net:+.4%}   95% CI [{o.ci_lo:+.4%}, {o.ci_hi:+.4%}]')
    print(f'  retained       {o.retained_frac:.1%}')
    print(f'  hit rate       {o.hit_rate:.1%}')
    print(f'  Sharpe         {o.sharpe:.2f}')
    print(f'  Deflated SR    {A["oos_dsr"]:.3f}   (P that the Sharpe survives the search)')
    print(f'  max drawdown   {o.max_dd:.1%}')
    print()
    verdict = 'REJECTED' if o.ci_lo > 0 else 'NOT REJECTED'
    print(f'  H0 (no profitable veto set): {verdict}')


### Deflated Sharpe: pricing in the search

The expected maximum Sharpe under the null **grows with the number of strategies
tried**. Comparing an observed Sharpe against zero after searching thousands of
veto sets is the textbook backtest-overfitting error.

The Deflated Sharpe Ratio (Bailey & Lopez de Prado, 2014) compares it against
that expected maximum instead, and additionally corrects for skew and kurtosis --
which matter enormously here, since a handful of huge winners inflate the sample
Sharpe in a way the Gaussian formula does not penalise.


In [ ]:
curves = {'all candidates': A['baseline_curve']}
if 'oos_curve' in A and not A['oos_curve'].empty:
    curves['walk-forward OOS'] = A['oos_curve']
viz.equity_curves(curves).show()


In [ ]:
oos = A.get('oos_trades')
if oos is not None and not oos.empty:
    viz.return_distributions(
        A['trades_sample']['net_return'], oos['net_return'],
        filtered_label='after veto (out-of-sample)',
    ).show()


## Veto interactions

Vetoes are not independent -- `veto_thin_adv` and `veto_wide_spread` both proxy
illiquidity, so combining them adds little beyond either alone. The diagonal is
each veto in isolation; the midpoint of the colour scale is the no-veto baseline.


In [ ]:
from research.penny.features import veto as veto_mod
names = [v for v in veto_mod.veto_names(False)][:10]
viz.veto_interaction(A['search'], names, A['baseline'].mean_net).show()


In [ ]:
# Match on (symbol, date), not the index: oos_trades was concatenated with
# ignore_index=True, so its positional index would collide spuriously.
base = A['trades_sample'].copy()
oos = A.get('oos_trades')
if oos is not None and not oos.empty:
    kept = set(zip(oos['symbol'], oos['date']))
    base['population'] = [
        'after veto' if k in kept else 'all candidates'
        for k in zip(base['symbol'], base['date'])
    ]
    viz.outcome_mix(base, 'population').show()


## The sealed holdout

The final slice of sessions was removed **before any tuning** and is opened
exactly once, here. Nothing in the parameter search, the veto search, or the
walk-forward ever saw it.

This is the number that matters. Everything else was, in one way or another,
fitted.


In [ ]:
h, hb = A.get('holdout'), A.get('holdout_baseline')
if h is not None:
    print(f"veto set applied : {', '.join(A['holdout_vetoes']) or '(none)'}")
    print()
    print(f'{"":<18}{"baseline":>14}{"after veto":>14}')
    print(f'{"trades":<18}{hb.n_trades:>14,}{h.n_trades:>14,}')
    print(f'{"E[net]":<18}{hb.mean_net:>13.3%}{h.mean_net:>14.3%}')
    print(f'{"CI low":<18}{hb.ci_lo:>13.3%}{h.ci_lo:>14.3%}')
    print(f'{"CI high":<18}{hb.ci_hi:>13.3%}{h.ci_hi:>14.3%}')
    print(f'{"hit rate":<18}{hb.hit_rate:>13.1%}{h.hit_rate:>14.1%}')
    print(f'{"Sharpe":<18}{hb.sharpe:>14.2f}{h.sharpe:>14.2f}')
    print()
    if h.ci_lo > 0:
        print('Holdout interval excludes zero: H1 survives its final test.')
    else:
        print('Holdout interval includes zero: H0 is NOT rejected on unseen data.')


## Capacity

A percentage return is not a business. At 1% participation in the day's dollar
volume, the dollar profit per day bounds what this could ever be worth -- and in
microcaps that bound is the binding constraint, not the edge.

This is the mechanism Hou, Xue & Zhang (2020) identify: microcaps are ~3% of
market value but ~60% of listed names, and anomalies concentrate there precisely
*because* they cannot be traded away.


In [ ]:
oos = A.get('oos_trades')
if oos is not None and not oos.empty and 'adv_20' in oos.columns:
    cap = A['params']['barriers']['participation_cap']
    size = oos['adv_20'] * cap
    pnl = size * oos['net_return']
    per_day = pnl.groupby(oos['date']).sum()
    print(f'participation cap      {cap:.0%} of trailing 20-day dollar volume')
    print(f'median position size   ${size.median():,.0f}')
    print(f'median daily P&L       ${per_day.median():,.0f}')
    print(f'mean daily P&L         ${per_day.mean():,.0f}')
    print(f'annualised (252d)      ${per_day.mean() * 252:,.0f}')


## Barrier geometry

Geometry cannot be tuned by the ordinary loop -- it is baked into the labels, so
each candidate needs a full dataset rebuild. A +30%/-10% target needs a 25% win
rate merely to break even *before* costs; the observed rate was 23.6%. Lowering
the target raises the hit rate but shrinks the payoff, so there is a real optimum
rather than a monotone improvement to chase.

**The decisive column is `trade_mean_gross`.**


In [ ]:
sweep = pd.read_csv(ROOT / 'data/research_penny/barrier_sweep.csv')
cols = ['profit_target', 'stop_loss', 'breakeven_win_rate', 'objective',
        'trade_mean_net', 'trade_mean_gross', 'p_target', 'p_stop']
display(sweep[cols].sort_values('objective', ascending=False).style.format({
    'profit_target': '{:.0%}', 'stop_loss': '{:.0%}', 'breakeven_win_rate': '{:.1%}',
    'objective': '{:+.5f}', 'trade_mean_net': '{:+.2%}', 'trade_mean_gross': '{:+.2%}',
    'p_target': '{:.1%}', 'p_stop': '{:.1%}',
}))
print('\nGross return is negative at EVERY geometry tested.')
print('This is not only a cost story -- there is no directional edge to harvest')
print('at any reward/risk ratio.')


## Conclusions

### H0 is not rejected

After an exhaustive search over 299 veto sets, a barrier sweep over six
geometries, and eight tuning iterations, **no veto set produces a profitable
population** -- in sample, out of sample, or on the sealed holdout.

| | E[net] per trade | 95% CI |
|---|---|---|
| baseline (no veto) | **-4.71%** | [-4.85%, -4.57%] |
| best walk-forward OOS | **-4.28%** | [-4.54%, -4.03%] |
| **sealed holdout** | **-4.48%** | [-4.81%, -4.15%] |

- veto sets passing one-sided FDR: **0 of 299**
- Deflated Sharpe Ratio: **0.000**
- all three null tests: **PASS**

### What *is* true

The vetoes work in the sense Lemma 1 describes. They identify genuinely worse
sub-populations, consistently and stably: `veto_thin_adv` was selected in every
fold, `veto_high_max` in most, and the improvement is a reliable ~0.4-0.7
percentage points. Selection degradation from train to test was **0.0036** --
the vetoes are not overfitted artefacts.

They are simply nowhere near large enough, which is Lemma 2.

### Why, mechanically

**1. There is no gross edge.** At the best geometry the mean *gross* return is
**-0.39%** -- negative before a single basis point of cost, and negative at every
geometry tested. Buying a gapping microcap at 09:35 and exiting the same day is
not a positive-expectation bet that frictions spoil; it is a negative-expectation
bet that frictions make much worse.

**2. Costs then remove another ~4.4 points.** Mean round-trip cost is **437 bps**,
estimated from the opening window's own high-low range plus square-root impact at
1% participation. This is exactly the Avramov/Chordia/Goyal and
Novy-Marx/Velikov mechanism: the effects live where the frictions are largest.

**3. One trade in seven is not enterable at all.** 2,977 of 20,367 candidates
(14.6%) had **no print** in the decision minute. On paper they are opportunities;
in reality there was nothing to buy.

### What would change the answer

This result is specific to what was tested, and three things could move it:

- **The short side.** Every effect found here points the same way -- high-MAX,
  illiquid, wide-spread names underperform. That is a *short* thesis, and the
  literature (Beneish, Lee & Nichols 2015) says the returns are real but locked
  behind borrow availability. Testing it needs the borrow data the companion
  recorder is now accruing.
- **Filing-based vetoes.** Pending dilution (S-3/424B5), Nasdaq deficiency
  notices (8-K 3.01) and going-concern language are all free from EDGAR and were
  *not* in this run. They target the mechanism most directly.
- **A cheaper entry.** All the cost here is paid crossing the spread at 09:35. A
  resting-limit entry would change the cost structure entirely -- at the price of
  non-fills, which this dataset can measure.

### What would not change it

More vetoes of the kind already tested, a bigger search, or a machine-learning
model on these features. The binding constraint is not classifier quality; it is
that the underlying population has negative gross expectancy. No filter over
these features fixes that, and a larger search only makes the multiple-testing
correction harsher.

> **The pre-registered kill criterion was:** "no veto set with net E[r] > 0 at 95%
> after FDR correction, with >= 2% of the population retained -> null not
> rejected, stop and report the negative result." That criterion is met, and this
> is that report.



---

# Round 2 — can a learned ranker close the gap?

Round 1 tested a *veto filter*: twelve binary thresholds, searched exhaustively.
That discarded most of the available information, because the underlying signals
are continuous and correlated — `obs_range` carries an IC of −0.18 against the
barrier-free return, but "is `obs_range` above 0.20" carries a fraction of it.

Round 2 replaces the vetoes with a **meta-labelling model** (López de Prado 2018,
ch. 3): the candidate screen is the primary rule, and a LightGBM secondary model
predicts whether each signal will clear costs. The target is binary — *would this
trade have cleared the bar* — not the return, because a regressor would spend its
capacity fitting the enormous right tail, which is the part least likely to repeat.

It also adds the one feature family that observes the **supply mechanism directly**
rather than proxying it through price: SEC filings.

### A diagnostic that had to be thrown away

The model's score-decile monotonicity reads **ρ = +0.84**, which looks like strong
evidence. Under shuffled labels the same statistic averages **|ρ| = 0.515** and
reaches **+0.709** — with ten bins, Spearman ρ has a standard deviation near 0.33.

So ρ is not evidence, and it is not used as such below. Everything rests on
`compare_selection_to_baseline`: a paired, day-clustered bootstrap of the return
difference between the selected population and taking everything.


In [ ]:
with open(ROOT / 'data/research_penny/analysis_model.pkl', 'rb') as fh:
    M = pickle.load(fh)

print(f"trades   : {M['n_trades']:,}")
print(f"features : {M['n_features']}  (EDGAR included: {M['use_edgar']})")
print(f"trials   : {M.get('n_trials', 0)}")
print()
for line in M['folds']:
    print(line)


### The grid

Each row is one (selectivity × cost bar) configuration, scored strictly out of
sample. `lift_gross` is the paired improvement over taking every candidate;
`meets_h1` is the pre-registered criterion.


In [ ]:
g = M['grid']
cols = ['fraction', 'cost', 'n', 'mean_gross', 'gross_ci_lo',
        'lift_gross', 'lift_p', 'mean_net', 'net_ci_lo', 'meets_h1']
display(g[cols].style.format({
    'fraction': '{:.0%}', 'cost': '{:.2%}', 'mean_gross': '{:+.3%}',
    'gross_ci_lo': '{:+.3%}', 'lift_gross': '{:+.3%}', 'lift_p': '{:.3f}',
    'mean_net': '{:+.3%}', 'net_ci_lo': '{:+.3%}',
}))
print(f"\nbest gross          : {g['mean_gross'].max():+.4f}")
print(f"best net CI low     : {g['net_ci_lo'].max():+.4f}")
print(f"H1 met at 437bps    : {M['h1_met_at_assumed_cost']}")
print(f"H1 met at ANY cost  : {M['h1_met_any_cost']}")


In [ ]:
table = M['decile_tables']['0.0437']
viz.model_score_deciles(table, M['cost_levels']).show()


### Did the filing features help?

This is round 2's one genuine positive finding. Price and volume features all
proxy the same latent quantity — how extreme the name has been — so combining
them adds little. Filings observe something different.

| features | best gross | lift vs baseline | p |
|---|---|---|---|
| price/volume only | +0.14% | +1.17pp | 0.025 |
| **+ EDGAR filings** | **+1.51%** | **+2.54pp** | 0.032 |

Roughly a tenfold increase in achievable gross. In the candidate population,
**18.6%** carry a listing-deficiency notice (8-K item 3.01) within 180 days and
**48.9%** a dilutive 8-K — these are common events, not rare ones.

Every filing feature is keyed on `acceptanceDateTime`, never `filingDate`: a
filing accepted at 13:00 ET must not inform a 09:35 decision that morning.


In [ ]:
imp = M['importances'].get('0.0437')
if imp is not None and not imp.empty:
    top = imp.mean(axis=1).sort_values(ascending=False).head(15)
    display(top.to_frame('mean importance'))


### The regime finding, and why it fails

The candidate rate runs ~3.5/session before 2020 and ~15.5 in 2025–26, so the
population is not stationary. Splitting by trailing 60-day candidate rate, edge
appears to concentrate in busy regimes — the top decile there returns **+2.11%**
gross, which clears the *optimistic* 200 bps bar.

It does not survive:

- **no CI excludes zero** at any selectivity (best p = 0.056, uncorrected)
- **only 2 of 4 folds contain busy-regime days**, and they disagree: +0.34% vs +3.00%
- **Deflated Sharpe = 0.127** at 27 trials, against a raw Sharpe of 1.52

A single period (2024–26) drives it. This is what an overfit looks like when the
machinery is honest enough to say so — and it is why the Deflated Sharpe is in
this notebook at all.


In [ ]:
r = M.get('regime')
if r is not None and not r.empty:
    display(r.style.format({
        'gross_all': '{:+.3%}', 'gross_top10': '{:+.3%}',
        'top10_ci_lo': '{:+.3%}', 'top10_ci_hi': '{:+.3%}', 'lift': '{:+.3%}',
    }))
    viz.regime_comparison(r).show()


## Round 2 verdict

**H1 is not met at any tested cost level (200 / 300 / 437 bps).**

| | |
|---|---|
| baseline gross | −1.08% |
| best model gross (with EDGAR) | **+1.51%** |
| best gross 95% CI | [−0.56%, …] — includes zero |
| best net @ 437 bps | −2.86% |
| best net @ 200 bps (optimistic) | −1.74% |

The model is real. The lift is significant at p = 0.001–0.03 across
configurations and moves gross from −1.08% to +1.51% — a genuine **+2.6
percentage points**. It is still not enough against a friction cost of at least
200 bps and probably 437.

**Two rounds now agree by different mechanisms.** No veto set works; a learned
ranker with filing data works *as a ranker* and cannot reach the bar. The
binding constraint is that the candidate population's gross expectancy is about
−1%, and the best available selection buys roughly 2.5 points of that back.

### The one experiment that could still change the answer

**Measure the cost bar.** It has never been measured — it is
`2 × (118 bps half-spread + 100 bps impact)`, both by convention. At +1.51%
gross, this strategy is *profitable* if true round-trip cost is below ~150 bps.

That is no longer a footnote; it is the decisive open question, and it is
answerable in a day from historical NBBO quotes on the same free tier that
supplied everything else here.

One caveat that cuts the other way: the literature on limit-order execution is
explicit that passive fills suffer adverse selection — *if a buy limit doesn't
fill, the price has typically moved higher*. So a cheaper entry is not free, and
the net effect has to be measured rather than assumed.


---

# Round 3 — measured costs, four new data sources, sequence models

Round 2 failed H1 against a cost bar of **437 bps that nobody had measured**.
Round 3 measures it, adds the four remaining free data sources, and tests whether
a sequence model over the pre-open path can find what aggregates destroy.

## Gate A — the cost bar

| component | assumed | **measured** (n=9,794) |
|---|---|---|
| half-spread | 118 bps | **37.7 bps** |
| impact @1% ADV | 100 bps | **71.8 bps** |
| **depth shortfall** | **0 bps** | **398.4 bps** |
| **round trip** | 437 bps | **728.9 bps** |

The assumption was wrong in both directions and they partly cancelled. Spread was
~3× too harsh; **depth was ignored entirely**, and it dominates. The median inside
quote is ~\$1,900 while 1% of ADV is ~\$19,000 — the order walks the book.

Cost is therefore a **function of order size**, which no previous round modelled:
\$2,000 → ~162 bps, \$10,000 → ~387 bps, \$25,000 → ~739 bps, with an
**irreducible floor of 77 bps** paid at any size.


In [ ]:
with open(ROOT / 'data/research_penny/analysis_round3.pkl', 'rb') as fh:
    R3 = pickle.load(fh)

import json
cost = json.load(open(ROOT / 'data/research_penny/cost_summary.json'))
print(f"measured on {cost['n_matched']:,} symbol-days")
for k in ['half_spread', 'impact', 'depth', 'round_trip']:
    key = f'measured_{k}_bps_median'
    if key in cost:
        print(f'  {k:<12} {cost[key]:>7.1f} bps')
print()
for line in R3['folds']:
    print(line)


## Adding features made the model worse

The ablation is monotonically **degrading**, and **Gate B fails** on a −0.12%
premarket lift.

This contradicts round 2, where EDGAR appeared to give a 10× improvement. The
difference is the **fold restructuring**: round 2's fold 0 trained on **610 rows**,
round 3's on **2,682**. With a sane baseline the base model scores +0.71% instead
of +0.14%, and the apparent EDGAR benefit largely disappears.

> **Round 2's headline improvement was substantially an artifact of a degenerate
> fold.** Fixing the fold structure was worth more than any feature added since.

The mechanism is ordinary feature bloat: 33 → 60 features against 2,682–10,613
training rows buys more variance than signal.


In [ ]:
g3 = R3['grid']
cols = ['feature_set', 'fraction', 'n', 'gross', 'gross_lo', 'lift', 'lift_p',
        'net_2000', 'net_lo_2000']
display(g3[g3['fraction'] == 0.10][cols].style.format({
    'fraction': '{:.0%}', 'gross': '{:+.3%}', 'gross_lo': '{:+.3%}',
    'lift': '{:+.3%}', 'lift_p': '{:.4f}',
    'net_2000': '{:+.3%}', 'net_lo_2000': '{:+.3%}',
}))
print(f"\nGate B: premarket lift {R3['premarket_lift']:+.4f} -> "
      f"{'PASS' if R3['gate_b_pass'] else 'FAIL'}")


## The sequence models — the one thing that helped

Gate B failed, so under the pre-registered protocol the encoders should not have
been built. They were run anyway as an **explicitly exploratory arm**, excluded
from the H1 family and from the multiple-testing correction.

The **transformer blend reaches +1.22% gross with a CI lower bound of +0.14%** —
the first configuration in the entire project whose gross interval excludes zero
(lift +2.10pp, p<0.0001).

The pre-open path evidently carries **shape** information that ~16 hand-made
aggregates destroy. That is exactly the argument for a sequence model *on a
sequence*, and exactly why it was **not** justified on the tabular features:
Grinsztajn et al. (2022) find tree ensembles still win there at ~10k samples,
and this study has 17k.

It remains exploratory. Gate B was pre-registered and it failed; reporting a
post-hoc arm as confirmatory would be the error this whole apparatus exists to
prevent.


In [ ]:
seq = g3[g3['feature_set'].str.startswith('blend:') | (g3['feature_set'] == 'base')]
display(seq[seq['fraction'] == 0.10][cols].sort_values('gross', ascending=False)
        .style.format({
            'fraction': '{:.0%}', 'gross': '{:+.3%}', 'gross_lo': '{:+.3%}',
            'lift': '{:+.3%}', 'lift_p': '{:.4f}',
            'net_2000': '{:+.3%}', 'net_lo_2000': '{:+.3%}',
        }))


## Round 3 verdict

**H1 is not met at any order size.**

| | |
|---|---|
| best gross (exploratory) | **+1.22%** — CI [+0.14%, …] |
| best gross (pre-registered) | +0.71% — CI includes zero |
| net @ \$2,000 | **−0.90%** |
| net @ \$5,000 | −3.47% |
| net @ \$10,000 | −6.95% |
| H1 family | 45 cells, BH-FDR corrected |
| cumulative trials (rounds 2+3) | 72 |

The measured cost floor is **~162 bps** at a \$2,000 order. The best gross the
project has produced is **122 bps**, and that from an exploratory arm.

**Even at the smallest tradable size, with the best model, gross does not reach
the cost floor.**

### What three rounds have established

1. **The explosions are real** — 1.45% of candidates double intraday.
2. **Selection works, at a scale that does not matter.** Vetoes, a GBDT, and a
   transformer all find genuine signal worth roughly +1 to +2 percentage points
   of gross. Costs are 1.6 to 7.3 points.
3. **Exit policy, barrier geometry and feature count are not levers.** Nine exit
   policies, six geometries and five feature sets were tested; none changed the
   conclusion, and more features actively hurt.
4. **The binding constraint is microstructure, not prediction.** The median inside
   quote is ~\$1,900. That is the whole story.


---

# Round 4 — precision inside the liquid band

## Why round 3's conclusion was premature

Round 3 measured a median round-trip cost of **728.9 bps**, dominated by a
**398.4 bps depth shortfall**, and concluded that the binding constraint is
microstructure rather than prediction. Two defects in that measurement made it
look more final than the evidence supports.

**The cost table sized every order at 1% of ADV.** A more liquid name therefore
received a proportionally *larger* order, and appeared more expensive. That is
not how the strategy would trade -- a real book places a fixed dollar amount.
Recomputed at a fixed order size, the ranking inverts:

| ADV bucket | n | RT @ \$2k | RT @ \$10k |
|---|---|---|---|
| $\leq$ \$300k | 1,501 | 395.1 bps | 1034.6 |
| \$300k–1M | 2,085 | 259.4 | 683.8 |
| \$1–5M | 2,731 | 149.4 | 371.6 |
| \$5–20M | 1,956 | **98.6** | 215.5 |
| > \$20M | 1,521 | **63.1** | **124.0** |

A **6.3× span**. The "irreducible 77 bps floor" reported in round 3 was a
whole-panel median, not a floor. **Cost is a filter, not a wall** -- and
`adv_20` is ex-ante, already in the panel, and free.

**The funnel was narrow and the cut shallow.** `gap >= +10%` with
`adv_20 >= $100k` gave 20,558 candidates, and selection took the top 10%. That
is a recall posture applied to a precision problem: the realistic strategy takes
very few trades and needs them to be right.

Round 4 therefore **widens the funnel into the liquid band** and **cuts far
deeper**, to 0.5%. `MIN_RETAINED` is deleted: recall is an output here, not a
setting.


## Making the cost accounting stricter first

Three defects in round 3's charge, each of which flattered the result. All are
fixed **before** the promising part, which is what separates this from motivated
reasoning.

**1. Depth was charged once for a round trip that crosses the book twice.**
`2 × (half_spread + impact) + depth` assumes the 15:55 exit fills entirely at
the inside. Since depth was 398 of the 728 bps, the single largest term was
half-counted. Each leg is now charged against its own measured book.

**2. Only the ask was ever measured.** A long exit sells into the *bid*, and in
this universe the two sides routinely differ by an order of magnitude.

**3. The estimate peeked at the entry minute.** Round 3 took the median quote
over `[09:35:00, 09:36:00)` while entry is the 09:35 bar *open*. Harmless while
cost was evaluation-only; look-ahead the moment cost informs the decision, which
is the whole of round 4. So the two quantities are now separated:

| | window | role |
|---|---|---|
| `cost_hat` | `[09:34:00, 09:35:00)` | **decision input** — what was knowable |
| `cost_realized` | 09:35 ask + 15:55 bid | **charged** — what actually happened |

Decide with what you knew; get charged what happened. A tripwire asserts that no
`cost_hat` input uses a quote at or after 09:35:00 — and it fired immediately on
live data, because Alpaca's `end` is an *inclusive* instant and returns the
09:35:00.000 quote for a `[09:34, 09:35)` request. That is the same trap that
made rounds 1–2 silently drop each target day's own bar.

**Coverage, stated plainly.** Round 3's cost table covered **56.3%** of trades
and *dropped* the rest from the bootstrap rather than defaulting them — a fact
recorded in the code (`run_round3_study.py:174`, "tilts the estimate ...
optimistically") but absent from its log. Round 4 measures every session, and
anything still unquoted is **filtered as untradeable** rather than dropped: you
cannot trade what you cannot quote, so those symbol-days are outside the
strategy by construction.


In [ ]:
with open(ROOT / 'data/research_penny/analysis_round4.pkl', 'rb') as fh:
    R4 = pickle.load(fh)

G4 = R4['grid']
print(f"pre-registration commit : {R4['preregistration_commit']}")
print(f"candidates (raw trades) : {R4['n_trades_raw']:,}")
print(f"period                  : {R4['date_min']} .. {R4['date_max']}")
print(f"confirmatory arms       : {', '.join(R4['confirmatory_arms'])}")
print(f"cells / eligible        : {R4['n_cells']} / {R4['n_eligible']}")
print(f"prior trials folded in  : {R4['prior_trials']}")


### The tradeability filter, and what it removed

A candidate is untradeable at a given order size if it has no firm inside quote,
a book smaller than the order, or a half-spread above 200 bps. The removed
population's own return is reported alongside the kept one — because a filter
that happens to strip out the losers is doing the work the model claims to do,
and that needs to be visible rather than assumed away.


In [ ]:
tr = pd.DataFrame(R4['tradeability']).T
tr.index.name = 'order_size'
display(tr[['n_before', 'n_after', 'share_kept', 'no_quote',
            'kept_gross_mean', 'removed_gross_mean']])


## The precision ladder

Three arms, declared in `PREREGISTRATION_R4.md` **before any result was
computed** (the commit hash is the timestamp):

| arm | rule |
|---|---|
| `gbdt_liquid` | top-*k* rank cut — the round-3 comparator, carried over unchanged |
| `ev_rule` | take when calibrated $\hat{p}\,E[r \mid win] - (1-\hat{p})E[\lvert r \rvert \mid lose] > \text{cost\_hat}_i$ |
| `blend_transformer` | promoted from round 3's exploratory arm, where it was the only configuration whose gross CI excluded zero |

The expected-value rule is the one that can **abstain**. A rank cut always takes
its top *k* whether or not anything clears its cost; on a day where no candidate
does, the correct number of trades is zero. That is why recall is reported as an
outcome rather than fixed in advance.

The label changed with it. Round 3 trained against a flat 437 bps while true
cost spanned 34 → 2000+ bps, so the model was learning the wrong question. Round
4 labels each trade against **its own** `cost_hat`.


In [ ]:
show = ['arm', 'fraction', 'order_size', 'n', 'n_days', 'retained',
        'gross', 'gross_lo', 'net', 'net_lo', 'cost_mean_bps', 'meets_h1']
display(G4[[c for c in show if c in G4.columns]]
        .sort_values(['order_size', 'arm', 'fraction']))


### Precision KPIs

A rule with a positive mean and eleven consecutive losers is not deployable,
however good its confidence interval looks. These are the numbers a low-recall
strategy is actually judged on, reported at **every** rung rather than only the
winner.


In [ ]:
kpi = [c for c in G4.columns if c.startswith('kpi_')]
cols = ['arm', 'fraction', 'order_size', 'kpi_n', 'kpi_hit_rate',
        'kpi_payoff_ratio', 'kpi_profit_factor', 'kpi_trades_per_month',
        'kpi_worst_day', 'kpi_max_consecutive_losers']
display(G4[[c for c in cols if c in G4.columns]].sort_values(
    'kpi_hit_rate', ascending=False).head(20))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
for arm, part in G4[G4['fraction'].notna()].groupby('arm'):
    p = part[part['order_size'] == part['order_size'].min()].sort_values('fraction')
    if p.empty:
        continue
    ax.plot(100 * p['fraction'], 100 * p['net'], marker='o', label=arm)
    ax.fill_between(100 * p['fraction'], 100 * p['net_lo'], 100 * p['net'],
                    alpha=0.15)
ax.axhline(0, color='k', lw=0.8, ls='--')
ax.set_xscale('log')
ax.set_xlabel('selectivity (% of candidates taken, log scale)')
ax.set_ylabel('net return per trade (%)')
ax.set_title('Precision ladder: does being pickier pay?')
ax.legend(fontsize=8)

ax = axes[1]
for size, part in G4.groupby('order_size'):
    p = part[part['fraction'].notna()].sort_values('fraction')
    if p.empty:
        continue
    ax.plot(100 * p['fraction'], p['cost_mean_bps'], marker='s',
            label=f'${int(size):,}')
ax.set_xscale('log')
ax.set_xlabel('selectivity (%)')
ax.set_ylabel('mean realised round-trip (bps)')
ax.set_title('What the selected trades actually cost')
ax.legend(title='order size', fontsize=8)

plt.tight_layout(); plt.show()


### Is the probability actually a probability?

The expected-value rule compares a model output to a cost in return units, so it
makes a claim the rank cut never had to: that $\hat{p}$ means something. A
calibrated model sits on the diagonal below. Isotonic regression is fitted
**inside each fold, on a chronologically later slice than the model itself** —
calibrating on the model's own training rows would learn from in-sample scores
that are systematically overconfident, and would fail silently rather than loudly.


In [ ]:
cal = R4.get('calibration', {})
if cal:
    size = sorted(cal)[0]
    tab = pd.DataFrame(cal[size])
    fig, ax = plt.subplots(figsize=(5.2, 5))
    ax.plot([0, tab[['predicted', 'realised']].to_numpy().max()],
            [0, tab[['predicted', 'realised']].to_numpy().max()],
            'k--', lw=0.8, label='perfect calibration')
    ax.plot(tab['predicted'], tab['realised'], marker='o', label='observed')
    ax.set_xlabel('predicted P(clears own cost)')
    ax.set_ylabel('realised frequency')
    ax.set_title(f'Calibration, ${int(size):,} order')
    ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()
    display(tab)
else:
    print('no calibration table recorded')


### Validity: the nulls, and sensitivity to the depth model

Both nulls run through the **full** pipeline including the calibrator, because
the calibrator is new this round and is exactly the sort of component that can
manufacture an edge out of nothing.

They are measured as **lift over the panel baseline**, never as a level. Round 3
recorded a shuffled-label null that "failed" for the wrong reason — it compared
the selected mean against zero on a panel whose own mean was far from zero, and
so flagged a null that was behaving perfectly.

The depth model is crude by its own docstring: one extra half-spread per
notional-multiple beyond the inside, with no depth-of-book data to calibrate the
shape. A conclusion that survives only at `levels_penalty = 1.0` is a conclusion
about that assumption, not about the market.


In [ ]:
nulls = pd.DataFrame(R4.get('nulls', []))
if not nulls.empty:
    display(nulls)
    print('all nulls pass:', bool(nulls['passes'].all()))
else:
    print('nulls not run')

sens = pd.DataFrame(R4.get('cost_sensitivity', []))
if not sens.empty:
    display(sens)


## Sources

**Method**

- Lopez de Prado, M. (2018). *Advances in Financial Machine Learning*, ch. 3
  (triple-barrier labelling), ch. 7 (purged K-fold, embargoing). Wiley.
- Bailey, D. & Lopez de Prado, M. (2014). "The Deflated Sharpe Ratio".
  *Journal of Portfolio Management* 40(5).
- Benjamini, Y. & Hochberg, Y. (1995). "Controlling the False Discovery Rate".
  *JRSS B* 57(1).
- Corwin, S. & Schultz, P. (2012). "A Simple Way to Estimate Bid-Ask Spreads
  from Daily High and Low Prices". *Journal of Finance* 67(2).

**The vetoes**

- Bali, T., Cakici, N. & Whitelaw, R. (2011). "Maxing Out: Stocks as Lotteries
  and the Cross-Section of Expected Returns". *JFE* 99(2). -- the MAX effect.
- Avramov, D., Chordia, T. & Goyal, A. (2006). "Liquidity and Autocorrelations
  in Individual Stock Returns". *Journal of Finance* 61(5).
- Frieder, L. & Zittrain, J. (2007). "Spam Works: Evidence from Stock Touts".
  *Hastings Comm. & Ent. Law Journal* 30(3).
- Beneish, M., Lee, C. & Nichols, C. (2015). "In Short Supply: Short Sellers and
  Stock Returns". *Journal of Accounting and Economics* 60(1).
- Hillion, P. & Vermaelen, T. (2004). "Death Spiral Convertibles". *JFE* 71(2).

**Why the baseline is what it is**

- Eraker, B. & Ready, M. (2015). "Do investors overpay for stocks with
  lottery-like payoffs?". *JFE* 115(3).
- White, J. (2016). "Outcomes of Investing in OTC Stocks". SEC DERA white paper.
- Hou, K., Xue, C. & Zhang, L. (2020). "Replicating Anomalies". *RFS* 33(5).
- Novy-Marx, R. & Velikov, M. (2016). "A Taxonomy of Anomalies and Their Trading
  Costs". *RFS* 29(1).
- Barber, B., Lee, Y., Liu, Y. & Odean, T. "Do Day Traders Rationally Learn About
  Their Ability?"
- Chague, F., De-Losso, R. & Giovannetti, B. (2020). "Day Trading for a Living?"
